In [1]:
!pip install torch torchvision torchtext

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 63.5 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import re

In [3]:
pairs = [
    ("hello", "hi how are you"),
    ("how are you", "i am fine"),
    ("what is your name", "i am a chatbot"),
    ("where are you from", "i am from internet"),
    ("bye", "goodbye")
]

In [4]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z]", " ", text)
    return text

In [5]:
def tokenize(sentence):
    return sentence.split()

In [6]:
vocab = set()

for q, a in pairs:
    vocab.update(tokenize(clean_text(q)))
    vocab.update(tokenize(clean_text(a)))

vocab = list(vocab)
word2idx = {word: i+1 for i, word in enumerate(vocab)}
idx2word = {i: word for word, i in word2idx.items()}

In [7]:
def encode(sentence):
    return [word2idx[word] for word in tokenize(clean_text(sentence))]

In [8]:
X = [encode(q) for q, _ in pairs]
Y = [encode(a) for _, a in pairs]

In [9]:
class Attention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attn = nn.Linear(hidden_size * 2, hidden_size)
        self.v = nn.Linear(hidden_size, 1)

    def forward(self, hidden, encoder_outputs):
        seq_len = encoder_outputs.size(0)
        hidden = hidden.repeat(seq_len, 1, 1)

        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))
        attention = self.v(energy).squeeze(2)

        return torch.softmax(attention, dim=0)

In [10]:
class ChatbotModel(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.encoder = nn.LSTM(embed_size, hidden_size)
        self.attention = Attention(hidden_size)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        outputs, (hidden, cell) = self.encoder(x)

        attn_weights = self.attention(hidden, outputs)
        context = torch.sum(attn_weights.unsqueeze(2) * outputs, dim=0)

        out = self.fc(context)
        return out

In [11]:
model = ChatbotModel(len(vocab)+1, 10, 16)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [14]:
for epoch in range(200):
    total_loss = 0

    for i in range(len(X)):
        inp = torch.tensor(X[i]).unsqueeze(1)
        target = torch.tensor([Y[i][0]])

        optimizer.zero_grad()
        output = model(inp)

        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss: {total_loss}")

Epoch 0, Loss: 15.583877801895142
Epoch 50, Loss: 0.028943269862793386
Epoch 100, Loss: 0.008329535950906575
Epoch 150, Loss: 0.004207278660032898


In [15]:
def predict(sentence):
    inp = torch.tensor(encode(sentence)).unsqueeze(1)
    output = model(inp)
    pred = torch.argmax(output).item()
    return idx2word.get(pred, "unknown")

In [16]:
print(predict("how are you"))

i


In [17]:
# Test prediction
input_text = "how are you"
output_text = predict(input_text)

print("Input:", input_text)
print("Output:", output_text)

# Save output to file
with open("output.txt", "w") as f:
    f.write(f"Input: {input_text}\n")
    f.write(f"Output: {output_text}")

# Auto download file
from google.colab import files
files.download("output.txt")

Input: how are you
Output: i


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>